In [3]:
# 3D windowed Plot
from ipywidgets import interact, FloatSlider, IntSlider
import matplotlib.pyplot as plt
import numpy as np

def plotw(volume, title="Recon Viewer", rotate=True , max_val=None,min_val=None, dynasdcm=False, norm=False):
    # 1. Move to CPU and handle complex data
    if hasattr(volume, 'get'): 
        volume = volume.get()
    
    # 2. Rotation to match Siemens Orientation (Optional but recommended)
    # This aligns with the 'R >> L' phase encoding in your protocol
    if rotate:
        volume=np.flip(volume,axis=-3)
        volume=np.rot90(volume,k=-1,axes=(1,2))
        volume=np.flip(volume,axis=-2)
    
    volume_abs = np.abs(volume)
    
    # # 3. CRITICAL: Remove NaNs/Infs for statistics
    # clean_data = volume_abs[np.isfinite(volume_abs)]
    
    # if clean_data.size == 0:
    #     print("Error: All values in volume are NaN or Inf.")
    #     return



    #Normalize and  Set the same dynamic range as siemens dicom if dynasdcm is set to True
    
    if norm:
        volume_abs_min=np.min(volume_abs)
        volume_abs_max=np.max(volume_abs)
        volume_abs=(volume_abs-volume_abs_min)/(volume_abs_max-volume_abs_min)
        if dynasdcm:
            max_val=1400
            min_val=0
            volume_abs*=max_val
        else:
            max_val=1
            min_val=0


    # 4. Calculate robust display limits
    # 99th percentile prevents "bright spot" artifacts from ruining contrast
    if max_val is None:
        # max_val = float(np.percentile(clean_data, 99))
        max_val = float(np.percentile(volume_abs, 99))
    if min_val is None:
        # min_val = float(np.percentile(clean_data, 1)) # Suggested noise floor
        min_val = float(np.percentile(volume_abs, 1)) # Suggested noise floor
    

    # if np.isnan(max_val) or max_val <= 0:
    #     max_val = 1.0 

    


    def update_view(slice_idx, v_min, v_max):
        # # Prevent v_min from being higher than v_max
        # if v_min >= v_max:
        #     v_min = v_max - (v_max * 0.01)

        plt.figure(figsize=(5, 5))
        # vmin and vmax control the windowing
        plt.imshow(volume_abs[slice_idx, :, :], cmap='gray', vmin=v_min, vmax=v_max)
        plt.title(f"{title} | Slice: {slice_idx}")
        plt.colorbar(label='Intensity')
        plt.axis('off')
        plt.show()

    # 5. Create sliders for both min (black level) and max (white level)
    interact(update_view, 
             slice_idx=IntSlider(min=0, max=volume.shape[0]-1, step=1, value=volume.shape[0]//2),
             v_min=FloatSlider(min=0, max=max_val, step=max_val/100, value=0, description='Min (Black)'),
             v_max=FloatSlider(min=max_val/100, max=max_val*3, step=max_val/100, value=max_val, description='Max (White)'))


In [7]:
#Dicom imagesabs
from pydicom import dcmread

file_dcm='/home/hpc/iwbi/iwbi112h/LLR_Daten/HighResMelon/dicom/100_wip_Snap_Cai_4_3_1-1-2_1p00_53_MR/9.dcm'

ds=dcmread(file_dcm).pixel_array
# print(np.shape(ds))

# plotw(ds,'Dicom ',rotate=False)

In [8]:
import h5py
import numpy as np

DATA_DIR='/home/vault/iwbi/iwbi112h/CEST_Data/'
# file='CEST_GRAPPA_recons_65.h5' # clipped grappa for 65 Reps 
file='CEST_GRAPPA_recons_Rep_idx_9_ker_8x8_prew.h5'
RO=224
REP=9
imgs=[]
RO_start=15 #start and end RO positin that cover FOV
RO_end=205
# RO_end=163
with h5py.File(DATA_DIR+file,'r') as f:
    # print(f.keys())
    for RO_idx in range(RO_start,RO_end): 
        # imgs.append(f[f'CEST_recon_RO_idx_{RO_idx}_8x8'][REP,:,:])
        imgs.append(f[f'CEST_recon_RO_idx_{RO_idx}'][0,...])

imgs=np.array(imgs)
print('imgs shape:',imgs.shape)
imgs=np.permute_dims(imgs,(1,2,0))
print('imgs shape:',imgs.shape)
# plotw(imgs,min_val=min_val,max_val=max_val)
# plotw(imgs,'Prewhiten_GRAPPA')
img_2=imgs[36,...]

imgs shape: (190, 72, 180)
imgs shape: (72, 180, 190)


In [9]:
# interpolate pixels to double the size 
import sigpy as sp
img_2_fft=sp.fft(img_2, axes=[-1,-2])
shape=img_2_fft.shape
print(shape)
img_2_fft=sp.resize(img_2_fft,(shape[-2]*2,shape[-1]*2))
print(img_2_fft.shape)
img_2_int=sp.ifft(img_2_fft,axes=[-1,-2])

# img_2_int=np.rot90(img_2_int,k=-1,axes=(-2,-1))
# img_2_int=np.flip(img_2_int,axis=-2)

# plotw(img_2_int[None, ...],'Zero Padded_GRAPPA')

(180, 190)
(360, 380)


In [10]:
from skimage.restoration import (denoise_bilateral,denoise_tv_chambolle ,denoise_wavelet)

# img_2_de=denoise_bilateral(abs(img_2_int),sigma_spatial=15, channel_axis=None)
img_2_de=denoise_tv_chambolle(abs(img_2_int),weight=2 ,channel_axis=None)
# img_2_de=denoise_wavelet(abs(img_2_int),channel_axis=None,method='BayesShrink', mode='soft',rescale_sigma=True)

plotw(img_2_de[None,...], 'Interp,denoised, GRAPPA')

interactive(children=(IntSlider(value=0, description='slice_idx', max=0), FloatSlider(value=0.0, description='…